In [23]:
from dotenv import load_dotenv
import os
import json
from openai import OpenAI
from PyPDF2 import PdfReader
import gradio as gr
import requests


In [24]:
load_dotenv(override=True)
openai = OpenAI()

In [25]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [26]:
def push(message):
    print(f"Sending push notification: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [27]:
push("HEY!!")

Sending push notification: HEY!!


In [28]:
def record_user_details(email, name="Name not provided", notes="not provided"):
  push(f"Recording interest from {name} with email {email} and notes: {notes}")
  return {"recorded": "OK"}

In [41]:
def record_unknown_questions(question):
  push(f"Recording unknown question about: {question}")
  return {"recorded": "OK"}

Tool called: record_user_details, flush=True
Sending push notification: Recording interest from Name not provided with email rengokukoh@gmail.com and notes: not provided


In [30]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [31]:
record_unknown_questions_json = {
    "name": "record_unknown_questions",
    "description": "Always use this tool to record any questions that couldn't be answered as you didn't know the answer,",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [32]:
tools = [{"type": "function", "function": record_user_details_json},
         {"type": "function", "function": record_unknown_questions_json}]

In [33]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional information about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_questions',
   'description': "Always use this tool to record any questions that couldn't be answered as you didn't know the answer,",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['qu

In [34]:
def handle_tool_calls(tool_calls):
  results = []
  for tool_call in tool_calls:
    tool_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    print(f"Tool called: {tool_name}, flush=True")
    
    if tool_name == "record_user_details":
      result = record_user_details(**arguments)
    elif tool_name == "record_unknown_questions":
      result = record_unknown_questions(**arguments)
    else:
      result = {"error": f"Unknown tool: {tool_name}"}
      
    results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
  
  return results

In [35]:
globals()["record_unknown_questions"]("this is a really hard question")

Sending push notification: Recording unknown question: this is a really hard question


{'recorded': 'OK'}

In [36]:
# More elegant way to write handle_tool_calls without if statements
def handle_tool_calls(tool_calls):
    results = []
    
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}, flush=True")
        
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {"error": f"Unknown tool: {tool_name}"}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    
    return results

In [37]:
reader = PdfReader("me/Profile.pdf")
linkedIn = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedIn += text

with open ("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()
    
name = "Bhagirath Parmar"

In [38]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedIn}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [39]:
def chat(message, history):
  messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
  done = False
  while not done:
    
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
    finish_reason = response.choices[0].finish_reason
    if finish_reason == "tool_calls":
      message = response.choices[0].message
      tool_calls = message.tool_calls
      results = handle_tool_calls(tool_calls)
      messages.append(message)
      messages.extend(results)
    else:
      done = True

  return response.choices[0].message.content


In [40]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool called: record_unknown_questions, flush=True
Sending push notification: Recording unknown question: Who is your favourite musician?
